In [1]:
# Installation rapide
!pip install polars s3fs boto3 pyarrow pandas openpyxl --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import boto3
import io
import pandas as pd
from io import BytesIO
import re

In [3]:
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)

In [4]:
bucket_name = "admindataanstat"
object_key = "Solde/DONNEES 2015/012015.xlsx"

In [5]:
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
data = pd.read_excel(BytesIO(obj['Body'].read()), engine='openpyxl')
# Affiche les 5 premières lignes pour un aperçu rapide.
data.head()

,MATRICULE||CODE_ORGANISME,MATRICULE,NOM,DATE NAISSANCE,SEXE,SITUATION MATRIMONIALE,NOMBRE ENFANT,LIEU AFFECTATION,SERVICE,ORGANISME,...,CLASSE/ECHELON,MONTANT BRUT,MONTANT NET,RETENUE PENSION,IMPOT,CHARGE PATRONALE,MOIS/ANNEE,GRADE 2,AGE RETRAITE,DATE RETRAITE
0,011242X15,011242X,DOSSO MEGBO,1939-01-01,Masculin,Marié,0.0,Seguela,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,NaN,34560,34145,0.0,415.0,0.0,12015,60,NaN,NaN
1,012856Q15,012856Q,KHISSY BEYNIOUAH FULBERT,1939-01-01,Masculin,Célibataire,0.0,Bouaké,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,NaN,42400,41582,0.0,509.0,0.0,12015,60,NaN,NaN
2,013454N15,013454N,VAHA TOMAS GNOMBLEI HENRI,1924-01-01,Masculin,Marié,0.0,Guiglo,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,NaN,45600,45053,0.0,547.0,0.0,12015,60,NaN,NaN
3,027861L25,027861L,CAPET YATO MATHIEU,1943-03-01,Masculin,Marié,0.0,Abidjan,A affecter,"Min. Affaires Etrangères, de l'Intégration Afr...",...,NaN,800000,662121,0.0,9600.0,0.0,12015,60,NaN,NaN
4,039659M38,039659M,COULIBALY YOUSSOUF,1945-12-01,Masculin,Marié,2.0,Abidjan,Dir Personnel Police Nationale,Min. de l'Intérieur et de la Sécurité,...,Commissaire divis. 1er éch,810000,810000,0.0,0.0,0.0,12015,A7,NaN,NaN


In [6]:
# Afficher la liste des variable de ma base de données
print(data.columns)

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE'],
      dtype='object')


In [7]:
# Créer une nouvelle colonne qui compte le nombre de position de la variable MATRICULE
data["nb_digits_mat"] = data["MATRICULE"].str.len()
print(data["nb_digits_mat"])

0         7
1         7
2         7
3         7
4         7
         ..
184885    7
184886    7
184887    7
184888    7
184889    7
Name: nb_digits_mat, Length: 184890, dtype: int64


In [8]:
# Filtrer les lignes où le code matricule ne tient pas sur 7 chiffres
codes_invalides = data[data["nb_digits_mat"] != 7]
print(codes_invalides[["MATRICULE", "nb_digits_mat"]])

Empty DataFrame
Columns: [MATRICULE, nb_digits_mat]
Index: []


In [9]:
# Nombre total de lignes dans la base
total_lignes = len(data)

# Nombre de valeurs uniques dans MATRICULE
valeurs_uniques = data["MATRICULE"].nunique()

# Vérification
if total_lignes == valeurs_uniques:
    print("✅ La variable 'MATRICULE' identifie de manière unique chaque observation.")
else:
    print("❌ La variable 'MATRICULE' n'est pas un identifiant unique.")

❌ La variable 'MATRICULE' n'est pas un identifiant unique.


In [10]:
# Nombre total de lignes dans la base
total_lignes = len(data)

# Nombre de valeurs uniques dans MATRICULE
valeurs_uniques = data["MATRICULE||CODE_ORGANISME"].nunique()

# Vérification
if total_lignes == valeurs_uniques:
    print("✅ La variable 'MATRICULE & CODE ORGANISME' identifie de manière unique chaque observation.")
else:
    print("❌ La variable 'MATRICULE & CODE ORGANISME' n'est pas un identifiant unique.")

✅ La variable 'MATRICULE & CODE ORGANISME' identifie de manière unique chaque observation.


In [11]:
# Sélectionner toutes les lignes où MATRICULE est dupliqué (apparaît plusieurs fois)
doublons = data[data.duplicated(subset="MATRICULE", keep=False)]

# Afficher la liste des doublons
print(doublons)

       MATRICULE||CODE_ORGANISME MATRICULE                            NOM  \
10                     054560DBQ   054560D                   MANZAN KONAN   
11                     054560DFA   054560D                   MANZAN KONAN   
48                     074062VBQ   074062V               KASSANYOU SALAMI   
49                     074062VFA   074062V               KASSANYOU SALAMI   
50                     074088YBQ   074088Y           TURQUIN TRAORE HENRI   
...                          ...       ...                            ...   
162004                 379614LMF   379614L     MENIN MESSOU BENIE MICHELE   
165967                 385079E33   385079E                  KONATE MOUSSA   
165968                 385079EBQ   385079E                  KONATE MOUSSA   
175753                 395032R24   395032R  NCHO-MOTTOH MARIE-PAULE BERNA   
175754                 395032RBQ   395032R  NCHO-MOTTOH MARIE-PAULE BERNA   

       DATE NAISSANCE      SEXE SITUATION MATRIMONIALE  NOMBRE ENFANT  \
10

In [12]:
# 1. Connexion au serveur
bucket_name = "admindataanstat"
prefixes = ["Solde/DONNEES 2015/", "Solde/DONNEES 2016/", "Solde/DONNEES 2017/", "Solde/DONNEES 2018/", "Solde/DONNEES 2019/", "Solde/DONNEES 2020/"]

In [16]:
# 2. Récupérer tous les fichiers Excel au format MMYYYY.xlsx ==========
import boto3
import re
from collections import defaultdict

# Connexion S3 (ajuste tes paramètres)
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)
bucket_name = "admindataanstat"
prefixes = ["Solde/DONNEES 2015/", "Solde/DONNEES 2016/", "Solde/DONNEES 2017/", "Solde/DONNEES 2018/", "Solde/DONNEES 2019/", "Solde/DONNEES 2020/"]

# Dictionnaire pour stocker tous les fichiers détectés par mois_annee
fichiers_par_mois = defaultdict(list)

# Parcourir tous les préfixes et récupérer les fichiers
for prefix in prefixes:
    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
    if "Contents" in response:
        for obj in response["Contents"]:
            key = obj["Key"]
            # Chercher fichiers Excel au format MMYYYY.xlsx
            match = re.search(r"(\d{6})\.xlsx$", key)
            if match:
                mois_annee = match.group(1)  # ex: '012015'
                fichiers_par_mois[mois_annee].append(key)

# Regrouper les mois par année (clé : année, valeur : set des mois)
fichiers_par_annee = defaultdict(set)
for mois_annee in fichiers_par_mois.keys():
    if len(mois_annee) == 6:
        mois = mois_annee[:2]
        annee = mois_annee[2:]
        fichiers_par_annee[annee].add(mois)

# Vérifier la complétude (12 mois) par année
for annee, mois_set in sorted(fichiers_par_annee.items()):
    mois_complets = set(f"{i:02d}" for i in range(1, 13))
    mois_manquants = sorted(mois_complets - mois_set)
    if mois_manquants:
        print(f"⚠️ Alerte : L'année {annee} ne contient que {len(mois_set)} fichiers au lieu de 12.")
        print(f"   Mois manquants : {', '.join(mois_manquants)}")
    else:
        print(f"✅ Année {annee} complète avec 12 fichiers.")

✅ Année 2015 complète avec 12 fichiers.
✅ Année 2016 complète avec 12 fichiers.
✅ Année 2017 complète avec 12 fichiers.
✅ Année 2018 complète avec 12 fichiers.
✅ Année 2019 complète avec 12 fichiers.
✅ Année 2020 complète avec 12 fichiers.


In [17]:
# 3. Charger les fichiers, ajouter la période, et concaténer
resultat_final = []

for mois, fichiers in fichiers_par_mois.items():
    for fichier in fichiers:
        try:
            # Lire le fichier Excel brut sans entête (pour ignorer les lignes vides)
            obj = s3_client.get_object(Bucket=bucket_name, Key=fichier)
            data_bytes = obj['Body'].read()
            df_raw = pd.read_excel(BytesIO(data_bytes), header=None)
            df_raw.dropna(how='all', inplace=True)

            # Identifier la bonne ligne d'en-tête
            colonnes_requises = {"MATRICULE||CODE_ORGANISME", "MATRICULE", "NOM", "DATE NAISSANCE","SEXE", "SITUATION MATRIMONIALE", "NOMBRE ENFANT", "LIEU AFFECTATION","SERVICE", "ORGANISME", "PRISE DE SERVICE", "SITUATION", "DATE DEBUT SITUATION", "EMPLOI", "FONCTION", "GRADE", "CLASSE/ECHELON", "MONTANT BRUT", "MONTANT NET", "RETENUE  PENSION", "IMPOT ", "CHARGE PATRONALE", 'MOIS/ANNEE', "GRADE 2", "AGE RETRAITE", "DATE RETRAITE"}
            for i, row in df_raw.iterrows():
                if colonnes_requises.issubset(set(row.astype(str))):
                    header_row_index = i
                    break
            else:
                print(f"⚠️ Aucune ligne d'en-tête détectée dans : {fichier}")
                continue  # ✅ ce `continue` est bien dans la boucle "for fichier in fichiers"

            # Relire avec les bons noms de colonnes
            df = pd.read_excel(BytesIO(data_bytes), skiprows=header_row_index)

            # Vérifier présence des colonnes
            if colonnes_requises.issubset(df.columns):
                df["PERIODE"] = mois
                df["DATE_COLLECTE"] = pd.to_datetime(mois, format="%m%Y")
                resultat_final.append(df)
                print(f"✅ {fichier} ajouté.")
            else:
                print(f"⚠️ Colonnes manquantes dans : {fichier} → {colonnes_requises - set(df.columns)}")

        except Exception as e:
            print(f"❌ Erreur avec {fichier} : {e}")

✅ Solde/DONNEES 2015/012015.xlsx ajouté.
✅ Solde/DONNEES 2015/022015.xlsx ajouté.
✅ Solde/DONNEES 2015/032015.xlsx ajouté.
✅ Solde/DONNEES 2015/042015.xlsx ajouté.
✅ Solde/DONNEES 2015/052015.xlsx ajouté.
✅ Solde/DONNEES 2015/062015.xlsx ajouté.
✅ Solde/DONNEES 2015/072015.xlsx ajouté.
✅ Solde/DONNEES 2015/082015.xlsx ajouté.
✅ Solde/DONNEES 2015/092015.xlsx ajouté.
✅ Solde/DONNEES 2015/102015.xlsx ajouté.
✅ Solde/DONNEES 2015/112015.xlsx ajouté.
✅ Solde/DONNEES 2015/122015.xlsx ajouté.
✅ Solde/DONNEES 2016/012016.xlsx ajouté.
✅ Solde/DONNEES 2016/022016.xlsx ajouté.
✅ Solde/DONNEES 2016/032016.xlsx ajouté.
✅ Solde/DONNEES 2016/042016.xlsx ajouté.
✅ Solde/DONNEES 2016/052016.xlsx ajouté.
✅ Solde/DONNEES 2016/062016.xlsx ajouté.
✅ Solde/DONNEES 2016/072016.xlsx ajouté.
✅ Solde/DONNEES 2016/082016.xlsx ajouté.
✅ Solde/DONNEES 2016/092016.xlsx ajouté.
✅ Solde/DONNEES 2016/102016.xlsx ajouté.
✅ Solde/DONNEES 2016/112016.xlsx ajouté.
✅ Solde/DONNEES 2016/122016.xlsx ajouté.
✅ Solde/DONNEES 

In [18]:
# ========== 4. Création du DataFrame panel ==========
if resultat_final:
    panel_solde_df = pd.concat(resultat_final, ignore_index=True)

    # Supprimer les lignes entièrement vides (optionnel, sécurité)
    panel_solde_df.dropna(how='all', inplace=True)

    print("✅ Fusion complète en format panel.")
    print(f"📊 Nombre total de lignes : {len(panel_solde_df)}")
    print(panel_solde_df.head())

    # (Facultatif) Export vers un fichier local
    # panel_solde_df.to_excel("panel_solde_df.xlsx", index=False)
    # panel_solde_df.to_csv("panel_solde_df.csv", index=False)

else:
    print("❌ Aucun fichier n'a été chargé correctement.")

✅ Fusion complète en format panel.
📊 Nombre total de lignes : 15627963
  MATRICULE||CODE_ORGANISME MATRICULE                        NOM  \
0                 011242X15   011242X                DOSSO MEGBO   
1                 012856Q15   012856Q   KHISSY BEYNIOUAH FULBERT   
2                 013454N15   013454N  VAHA TOMAS GNOMBLEI HENRI   
3                 027861L25   027861L         CAPET YATO MATHIEU   
4                 039659M38   039659M         COULIBALY YOUSSOUF   

        DATE NAISSANCE      SEXE SITUATION MATRIMONIALE  NOMBRE ENFANT  \
0  1939-01-01 00:00:00  Masculin                  Marié            0.0   
1  1939-01-01 00:00:00  Masculin            Célibataire            0.0   
2  1924-01-01 00:00:00  Masculin                  Marié            0.0   
3  1943-03-01 00:00:00  Masculin                  Marié            0.0   
4  1945-12-01 00:00:00  Masculin                  Marié            2.0   

  LIEU AFFECTATION                         SERVICE  \
0          Seguela   

In [19]:
import io

# Convertir en CSV
csv_buffer = io.StringIO()
panel_solde_df.to_csv(csv_buffer, index=False, encoding="utf-8-sig")

# Sauvegarder dans le bucket S3
s3_client.put_object(
    Bucket="admindataanstat",
    Key="Solde/panel_solde_df.csv",
    Body=csv_buffer.getvalue()
)

print("✅ Ta base a été sauvegardée dans le bucket S3 : Solde/panel_solde_df.csv")

✅ Ta base a été sauvegardée dans le bucket S3 : Solde/panel_solde_df.csv
